In [ ]:
import os
import pydicom 
from pydicom import dcmread
from math import *
import numpy as np
import SimpleITK as sitk
import sys, time, os
import re

## Sort DICOM files

In [ ]:
def clean_text(string):
    forbidden_symbols = ["*", ".", ",", "\"", "\\", "/", "|", "[", "]", ":", ";", " "]
    for symbol in forbidden_symbols:
        string = string.replace(symbol, "_")
    return string.lower()

def clean_folder_path(output_path):
    output_path = output_path.replace('\\','/')
    output_path = output_path.replace('<','_')
    output_path = output_path.replace('>','_')
    return output_path

def clean_filename(string):
    forbidden_symbols = ["*", ".", ",", "\"", "\\", "/", "|", "[", "]", ":", ";", " "]
    for symbol in forbidden_symbols:
        string = string.replace(symbol, "_")
    return string.lower()

def list_file_path(src):
    unsortedList = []
    for root, dirs, files in os.walk(src):
        for file in files:
            try:
                input_path = os.path.join(root, file)
                unsortedList.append(input_path)
            except:
                pass
    print('%s files found.' % len(unsortedList))
    return unsortedList

def sort_dcm_file(unsortedList,dst):
    i=0
    for dicom_loc in unsortedList:
        ds = pydicom.dcmread(dicom_loc, force=True)
        i=i+1
        patientID = clean_text(ds.get("PatientID", "NA"))
        studyDate = clean_text(ds.get("StudyDate", "NA"))
        seriesDescription = clean_text(ds.get("SeriesDescription", "NA"))
        modality = ds.get("Modality","NA")
        studyInstanceUID = ds.get("StudyInstanceUID","NA")
        seriesInstanceUID = ds.get("SeriesInstanceUID","NA")
        instanceNumber = str(ds.get("InstanceNumber","0"))
        fileName = modality + "." + seriesInstanceUID + "." + instanceNumber
        fileName = clean_filename(fileName)
        fileName = fileName+ ".dcm"
        
        try:
            if not os.path.exists(os.path.join(dst, patientID)):
                output_path = os.path.join(dst, patientID)
                output_path = clean_folder_path(output_path)
                os.makedirs(output_path)

            if not os.path.exists(os.path.join(dst, patientID, studyDate)):
                output_path = os.path.join(dst, patientID, studyDate)
                output_path = clean_folder_path(output_path)
                os.makedirs(output_path)

            
            if not os.path.exists(os.path.join(dst, patientID, studyDate, seriesDescription)):
                output_path = os.path.join(dst, patientID, studyDate, seriesDescription)
                output_path = clean_folder_path(output_path)
                os.makedirs(output_path)

            try :
                output_path=os.path.join(dst, patientID, studyDate, seriesDescription, fileName)
                output_path=clean_folder_path(output_path)
                os.renames(dicom_loc, output_path)
            except:
                print(str(dicom_loc))
                print('Error')
        except:
            print('Error creating files')

src = "XXX"   ## Import file
dst = "XXX"   ## Export file

###############################
unsortedList = list_file_path(src)
########################
sort_dcm_file(unsortedList,dst)

## DICOM to Nii conversion

In [ ]:
data_directory = "XXX"
save_result = "XXX"

def main(data_directory, save_result):
    Nimageouverte   = 0
    Nimagetraitees  = 0

    directory_list  = []
    i = 0
    
    for root, dirs, files in os.walk(data_directory):
        for subdirname in dirs:
            directory_list.append(os.path.join(root,subdirname))

    for i in range(len(directory_list)):
        data_directory = directory_list[i].replace('\\','/')
        series_IDs     = sitk.ImageSeriesReader.GetGDCMSeriesIDs(data_directory)
        if not series_IDs:
            print("ERROR: given directory \""+data_directory+"\" does not contain a DICOM series.")
        else:
            for i,series_ID in enumerate(series_IDs):   
                Nimageouverte     = Nimageouverte+1
                series_file_names = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(data_directory, series_ID,useSeriesDetails=False)
                try:
                    img_metadata = pydicom.dcmread(series_file_names[0])
                    if True :
                        try:
                            timeRMR1       = time.time()
                            Nimagetraitees = Nimagetraitees+1
                            series_reader  = sitk.ImageSeriesReader()
                            series_reader.SetFileNames(series_file_names)
                            img            = series_reader.Execute()
                            series_file_names  = series_file_names[0].split("/")
                            name               = series_file_names[-4]+"_"+series_file_names[-3]+"_"+series_file_names[-2]+"_.nii"
                            nameID             = series_file_names[-4]
                            save_path = os.path.join(save_result,name)
                            sitk.WriteImage(img,save_path)
                            timeRMR2               = time.time()
                            TimeForrunFunctionRMR2 = timeRMR2 - timeRMR1
                            print("\n")
                        except RuntimeError:
                            print ("--> Problem with image import and/or processing")
                except RuntimeError:
                    print ("--> Problem with reading metadata")
    print("\n")
    print("Number of images:"+str(Nimageouverte)+"\n")
    print("Number of processed images:"+str(Nimagetraitees)+"\n" )

main(data_directory, save_result)

# Rename files

In [ ]:
folder_path = "XXX"
pattern = re.compile(r"(?P<ref>[^_]+)_(?P<date>[^_]+)_(?P<suffix>.+)")

from collections import defaultdict
files_by_ref = defaultdict(list)

for filename in os.listdir(folder_path):
    match = pattern.match(filename)
    if match:
        ref = match.group("ref")
        date = match.group("date")
        suffix = match.group("suffix")
        files_by_ref[ref].append((date, suffix, filename))

In [ ]:
for ref, files in files_by_ref.items():
    unique_dates = sorted(set(date for date, _, _ in files))
    date_map = {}

    if len(unique_dates) == 3:
        date_map = {
            unique_dates[0]: "t0",
            unique_dates[1]: "t1",
            unique_dates[2]: "t2"
        }
    elif len(unique_dates) == 2:
        date_map = {
            unique_dates[0]: "t0",
            unique_dates[1]: "t1"
        }
    elif len(unique_dates) == 1:
        date_map = {
            unique_dates[0]: "t0"
        }

    for date, suffix, old_filename in files:
        new_filename = f"{ref}_{date_map[date]}_{suffix}"
        old_path = os.path.join(folder_path, old_filename)
        new_path = os.path.join(folder_path, new_filename)
        print(f"Rename: {old_filename} -> {new_filename}")
        os.rename(old_path, new_path)